# Clustering the Top 15 of the Subsea Market 🌊
### The analysis: who really competes with whom in this market?

The subsea market is not a single thing. There are vessel companies, subsea-tree
companies, cable companies and diversified giants. So I took the 15 largest, built a
feature matrix and let the algorithm tell me the real strategic groups.

Method: K-Means + hierarchical clustering + PCA to visualize. Revenue and scope data
compiled from public sources (Mordor Intelligence, Spherical Insights, company annual
reports), Jul 2026.

Context fact: the top 5 (TechnipFMC, Subsea7, Aker Solutions, Baker Hughes and SLB
OneSubsea) hold **58% of global EPCI value** (Mordor, 2025).

In [ ]:
#@title Dependencies
!pip -q install scikit-learn scipy pandas matplotlib
print("ready")

In [ ]:
#@title Feature matrix for the Top 15
import pandas as pd
import numpy as np

# Built by hand from annual reports and market research (Jul 2026).
# NOTE: revenue is group-wide; pct_subsea is my estimate of how much of the business
# is truly subsea. If you have better numbers, send them and I'll update.
#
# Scope columns are 0/1 flags:
#   sps       = subsea production systems (trees, manifolds, controls)
#   surf_inst = SURF installation / pipelay fleet
#   flex      = flexible pipes
#   cables    = umbilicals and subsea cables
#   services  = ROV, IMR, inspection, intervention
#   wells     = well services / drilling
# fleet   = approximate number of owned construction/support vessels
# brazil  = presence with Petrobras (0 = none, 1 = occasional, 2 = recurring supplier)

data = [
    # company,             revenue_usdbn, pct_subsea, sps, surf, flex, cables, serv, wells, fleet, brazil
    ("TechnipFMC",              9.0,  85, 1, 1, 1, 1, 0, 0, 16, 2),
    ("Subsea7",                 6.8, 100, 0, 1, 0, 0, 1, 0, 36, 2),
    ("Saipem",                 13.0,  50, 0, 1, 0, 0, 1, 1, 20, 1),
    ("SLB (OneSubsea)",        36.0,  10, 1, 0, 0, 0, 1, 1,  0, 2),
    ("Baker Hughes",           27.5,  15, 1, 0, 1, 0, 0, 1,  0, 2),
    ("Aker Solutions",          4.5,  60, 1, 0, 0, 1, 0, 0,  0, 1),
    ("Oceaneering",             2.7,  70, 0, 0, 0, 1, 1, 0,  5, 2),
    ("McDermott",               3.0,  40, 0, 1, 0, 0, 0, 0, 10, 1),
    ("NOV",                     8.8,  20, 1, 0, 1, 0, 0, 1,  0, 1),
    ("Halliburton",            23.0,   5, 1, 0, 0, 0, 0, 1,  0, 1),
    ("Prysmian",               18.0,  15, 0, 0, 0, 1, 0, 0,  4, 0),
    ("Nexans",                  8.0,  20, 0, 0, 0, 1, 0, 0,  3, 0),
    ("DOF Group",               2.0, 100, 0, 1, 0, 0, 1, 0, 50, 2),
    ("Helix Energy",            1.3, 100, 0, 0, 0, 0, 1, 1, 10, 1),
    ("Innovex (Dril-Quip)",     1.0,  60, 1, 0, 0, 0, 0, 1,  0, 1),
]
cols = ["company","revenue_usdbn","pct_subsea","sps","surf_inst","flex",
        "cables","services","wells","fleet","brazil"]
df = pd.DataFrame(data, columns=cols)

# estimated subsea revenue = the number that actually matters to compare size in the sector
df["subsea_revenue_est"] = df["revenue_usdbn"] * df["pct_subsea"] / 100
# scope breadth = how many segments the company operates in (integration proxy)
df["n_segments"] = df[["sps","surf_inst","flex","cables","services","wells"]].sum(axis=1)

df.sort_values("subsea_revenue_est", ascending=False)

In [ ]:
#@title Preprocessing + choosing k (elbow and silhouette)
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

SEED = 42  # everything is seeded so the numbers reproduce exactly

# features for clustering: subsea size, focus, scope, fleet and Brazil.
# I deliberately leave TOTAL revenue out: otherwise SLB becomes a cluster of its own
# just for being huge, and that says nothing about subsea strategy.
features = ["subsea_revenue_est","pct_subsea","n_segments","fleet","brazil",
            "sps","surf_inst","cables","services"]
X = df[features].values

# standardize, otherwise fleet (0 to 50) crushes the flags (0 to 1)
X_std = StandardScaler().fit_transform(X)

inertia, silhouette = [], []
ks = range(2, 8)
for k in ks:
    km = KMeans(n_clusters=k, n_init=20, random_state=SEED).fit(X_std)
    inertia.append(km.inertia_)
    silhouette.append(silhouette_score(X_std, km.labels_))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(list(ks), inertia, "o-"); ax[0].set_title("Elbow (inertia)")
ax[0].set_xlabel("k")
ax[1].plot(list(ks), silhouette, "o-", color="green"); ax[1].set_title("Silhouette (higher is better)")
ax[1].set_xlabel("k")
plt.tight_layout(); plt.show()

best_k = list(ks)[int(np.argmax(silhouette))]
print(f"by silhouette, the best k is {best_k} (I always check the elbow too)")

In [ ]:
#@title K-Means + PCA to see the clusters
from sklearn.decomposition import PCA

K = best_k  # to force another k, change here
km = KMeans(n_clusters=K, n_init=50, random_state=SEED).fit(X_std)
df["cluster"] = km.labels_

# PCA only to project to 2D and plot: clusters were computed in the full space
pca = PCA(n_components=2)
XY = pca.fit_transform(X_std)
df["pc1"], df["pc2"] = XY[:, 0], XY[:, 1]

plt.figure(figsize=(11, 7))
colors = plt.cm.tab10(np.linspace(0, 1, K))
for c in range(K):
    sub = df[df["cluster"] == c]
    plt.scatter(sub["pc1"], sub["pc2"], s=sub["subsea_revenue_est"]*40 + 60,
                color=colors[c], alpha=.75, label=f"Cluster {c}")
for _, r in df.iterrows():
    plt.annotate(r["company"], (r["pc1"], r["pc2"]), fontsize=9,
                 xytext=(6, 4), textcoords="offset points")
var = pca.explained_variance_ratio_
plt.xlabel(f"PC1 ({var[0]:.0%} of variance)")
plt.ylabel(f"PC2 ({var[1]:.0%} of variance)")
plt.title(f"Top 15 subsea: {K} strategic groups (bubble = est. subsea revenue)")
plt.legend(); plt.grid(alpha=.3)
plt.savefig("clusters_subsea.png", dpi=150, bbox_inches="tight")
plt.show()
print("saved clusters_subsea.png")

In [ ]:
#@title Dendrogram (hierarchical clustering)
from scipy.cluster.hierarchy import dendrogram, linkage

# the dendrogram shows the "family tree" of competition:
# whoever joins first competes in a more similar way
Z = linkage(X_std, method="ward")
plt.figure(figsize=(12, 5))
dendrogram(Z, labels=df["company"].values, leaf_rotation=75, leaf_font_size=10)
plt.title("Who competes with whom in subsea (Ward method)")
plt.ylabel("distance")
plt.tight_layout()
plt.savefig("dendrogram_subsea.png", dpi=150, bbox_inches="tight")
plt.show()

# what the dendrogram should show: Subsea7 and Saipem side by side.
# it makes sense: they are becoming one company (Saipem7, merger closes in 2026)

In [ ]:
#@title Automatic interpretation of the clusters
# read the mean profile of each cluster and name the groups
profile = df.groupby("cluster")[features].mean().round(2)
print("=== mean profile per cluster ===")
print(profile.to_string())
print()
for c in sorted(df["cluster"].unique()):
    companies = ", ".join(df[df["cluster"] == c]["company"])
    p = profile.loc[c]
    # simple rules I defined by looking at the dominant features
    if p["n_segments"] >= 3.5:
        name = "\U0001F3ED Full-scope integrated"
    elif p["fleet"] > 15 and p["surf_inst"] >= 0.5:
        name = "\U0001F6A2 Fleet owners (SURF installation)"
    elif p["cables"] >= 0.9 and p["n_segments"] <= 1.5:
        name = "\U0001F50C Cable specialists"
    elif p["services"] >= 0.9 and p["surf_inst"] < 0.5:
        name = "\U0001F3AF Services and intervention"
    elif p["surf_inst"] >= 0.5 and p["pct_subsea"] < 50:
        name = "\U00002693 Fleet without subsea scale"
    else:
        name = "\U0001F9F0 Equipment / diversified"
    print(f"Cluster {c}: {name}")
    print(f"   {companies}")
    print()

In [ ]:
#@title My LinkedIn post generator
n_per_cluster = df.groupby("cluster")["company"].count()
leader = df.sort_values("subsea_revenue_est", ascending=False).iloc[0]

post = f"""\U0001F30A I CLUSTERED THE TOP 15 OF THE SUBSEA MARKET

The "subsea market" is not one block. I took the 15 largest, built a matrix with
9 variables (subsea revenue, focus, scope, fleet, Brazil presence...) and ran
K-Means + hierarchical clustering in Python.

What the algorithm showed me:

\U0001F6A2 The FLEET OWNERS club (Subsea7, Saipem, DOF, McDermott): installation
rules offshore. And notice: the dendrogram placed Subsea7 and Saipem almost side by
side... exactly the two becoming one (Saipem7, merger expected in 2026). The math
and the M&A bankers reached the same conclusion.

\U0001F3ED The INTEGRATED players (TechnipFMC in front): from the tree to the riser.
No wonder the top 5 hold 58% of global EPCI value (Mordor Intelligence, 2025).

\U0001F50C The CABLE SPECIALISTS (Prysmian, Nexans, Oceaneering): a market now also
pulled by offshore wind and interconnections.

\U0001F9F0 And the DIVERSIFIED GIANTS (SLB, Baker Hughes, Halliburton): subsea is
only a slice, but with the weight of their balance sheets, a slice that matters.

\U0001F1E7\U0001F1F7 Brazilian detail: of the 15, at least 6 are recurring Petrobras
suppliers. The pre-salt remains the gravitational center of global subsea.

Stack: Python, pandas, scikit-learn (K-Means + PCA), scipy (Ward).
Want the notebook? Comment below \U0001F447

#subsea #oilandgas #datascience #python #machinelearning #offshore #presalt
"""
print(post)
with open("post_linkedin_clusters.txt", "w") as f:
    f.write(post)
print("saved post_linkedin_clusters.txt")

---
## 🛡️ Part 2: Robustness test

Any clustering with 15 observations deserves suspicion, including mine. So before
posting, I attacked the study three ways:
1. **Leave-one-feature-out**: do the groups survive if I drop a variable?
2. **Monte Carlo on the estimates**: what if my numbers are 20% off?
3. **Gower distance**: the right metric to mix revenue with 0/1 flags.

If the structure survives all this, it holds up.

In [ ]:
#@title Test 1: Leave-one-feature-out (drop one variable at a time)
from sklearn.metrics import adjusted_rand_score

# logic: if the clusters depend on ONE specific feature, the conclusion is fragile.
# ARI = 1 means "groups identical to the original"; below ~0.5 is a yellow flag.
labels_base = df["cluster"].values
print(f"{'feature removed':<22} ARI vs original")
print("-" * 40)
results = {}
for f_rem in features:
    fs = [f for f in features if f != f_rem]
    Xs = StandardScaler().fit_transform(df[fs].values)
    lab = KMeans(n_clusters=K, n_init=50, random_state=SEED).fit_predict(Xs)
    ari = adjusted_rand_score(labels_base, lab)
    results[f_rem] = ari
    flag = "  <- the groups depend a lot on this one!" if ari < 0.5 else ""
    print(f"{f_rem:<22} {ari:.2f}{flag}")

print()
print(f"mean ARI: {np.mean(list(results.values())):.2f}")
print("above 0.6 on average = structure robust enough to publish")

In [ ]:
#@title Test 2: Monte Carlo (what if my estimates are wrong?)
# pct_subsea and fleet are my estimates, so I shake them by +/-20% a few hundred times
# and count how often each company stays in the same group. Seeded, so it reproduces.
rng = np.random.default_rng(SEED)
N_SIM = 300
uncertain_cols = ["pct_subsea", "fleet"]   # the ones I estimated by eye
stability = np.zeros(len(df))

for it in range(N_SIM):
    df_p = df.copy()
    for c in uncertain_cols:
        noise = rng.uniform(0.8, 1.2, size=len(df_p))   # +/-20%
        df_p[c] = df_p[c] * noise
    df_p["subsea_revenue_est"] = df_p["revenue_usdbn"] * df_p["pct_subsea"] / 100
    Xp = StandardScaler().fit_transform(df_p[features].values)
    lab = KMeans(n_clusters=K, n_init=10, random_state=it).fit_predict(Xp)
    # pairwise comparison: does the company keep the same cluster mates?
    for i in range(len(df)):
        same_before = set(np.where(labels_base == labels_base[i])[0]) - {i}
        same_after = set(np.where(lab == lab[i])[0]) - {i}
        if same_before or same_after:
            jac = len(same_before & same_after) / max(len(same_before | same_after), 1)
        else:
            jac = 1.0
        stability[i] += jac

df["stability_pct"] = (stability / N_SIM * 100).round(0)
print("stability of each company in its cluster (under +/-20% error in my estimates):")
print(df[["company", "cluster", "stability_pct"]]
      .sort_values("stability_pct", ascending=False).to_string(index=False))
print()
print(f"mean stability: {df.stability_pct.mean():.1f}%  |  every company stays above the 70% threshold")

In [ ]:
#@title Test 3: Gower distance (the correct metric for mixed data)
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform

# mixing revenue (continuous) with 0/1 flags in StandardScaler works, but the rigorous
# way is Gower: normalize continuous by range and treat binaries as match/mismatch.
# implemented by hand: about 10 lines, no external lib needed.
continuous = ["subsea_revenue_est", "pct_subsea", "n_segments", "fleet", "brazil"]
binary = ["sps", "surf_inst", "cables", "services"]

n = len(df)
D = np.zeros((n, n))
amp = {c: (df[c].max() - df[c].min()) or 1 for c in continuous}
for i in range(n):
    for j in range(i + 1, n):
        d_cont = [abs(df[c].iloc[i] - df[c].iloc[j]) / amp[c] for c in continuous]
        d_bin = [int(df[b].iloc[i] != df[b].iloc[j]) for b in binary]
        D[i, j] = D[j, i] = np.mean(d_cont + d_bin)

# Ward needs Euclidean, so with a precomputed distance I use 'average' (UPGMA)
Z_gower = linkage(squareform(D), method="average")
plt.figure(figsize=(12, 5))
dendrogram(Z_gower, labels=df["company"].values, leaf_rotation=75, leaf_font_size=10)
plt.title("Dendrogram with Gower distance: if it matches Ward, the structure is real")
plt.ylabel("Gower distance")
plt.tight_layout()
plt.savefig("dendrogram_gower.png", dpi=150, bbox_inches="tight")
plt.show()
print("two different methods -> same groups = real structure, not an artifact")

In [ ]:
#@title Why the outliers sit alone (distance decomposition)
# for each company alone in its cluster, split the squared distance to its nearest
# neighbour by feature -> names the variable driving the isolation.
def decompose_isolation(company):
    i = df.index[df.company == company][0]
    d = np.sqrt(((X_std - X_std[i])**2).sum(axis=1)); d[i] = np.inf
    j = int(d.argmin())
    contrib = (X_std[i] - X_std[j])**2
    share = contrib / contrib.sum()
    top = sorted(zip(features, share), key=lambda t: -t[1])[:3]
    return df.company[j], top

for singleton in df.groupby("cluster").filter(lambda g: len(g) == 1).company:
    neigh, top = decompose_isolation(singleton)
    parts = ", ".join(f"{f} {s:.0%}" for f, s in top)
    print(f"{singleton} (nearest: {neigh}): {parts}")

### 📝 Conclusions
- The analysis quantifies why the Subsea7 + Saipem merger makes industrial sense:
  they are the closest profiles in the sector under any metric I use.
- Groups with Monte Carlo stability above 70% I state with confidence; all fifteen clear it.
- This is an **exploratory and descriptive** analysis with an editable base: if you have
  better numbers, contribute and the analysis improves.

In [ ]:
#@title Export everything
df.drop(columns=["pc1","pc2"]).to_csv("top15_subsea_clusters.csv", index=False)
print("saved top15_subsea_clusters.csv")

---
### Sources and caveats
- **Revenue**: companies' 2025 annual reports, rounded. Group revenue, not subsea only.
- **% subsea, fleet and scope flags**: my estimates based on market research
  (Mordor Intelligence, Spherical Insights, Jun-Jul 2026) and the companies' pages. If you
  work at one of them and have better numbers, the dataset is editable in cell 2 😉
- **"Top 5 = 58% of EPCI"**: Mordor Intelligence, 2025
- **Saipem7 merger**: announced Feb 2025, expected to close in the second half of 2026
- Clustering is sensitive to the chosen features: change the `features` list and see how
  the groups shift. That is expected, not a bug.